# 07 — Evaluation (Stage 15)

Run the full metric suite (PSNR, SSIM, LPIPS, and semantic mIoU where applicable) over the held-out test split for every trained model. **No numbers in this notebook are pre-filled — every value here must come from actually running `evaluate_model` against your checkpoints.**

In [ ]:
import sys; sys.path.append('/kaggle/working/SAR2EO-Diff')
%cd /kaggle/working/SAR2EO-Diff

## Evaluate U-Net

In [ ]:
!python scripts/evaluate.py --config configs/baseline_unet.yaml \
    --checkpoint checkpoints/unet/best.pt \
    --output results/tables/unet_test_metrics.json

## Evaluate Pix2Pix

In [ ]:
!python scripts/evaluate.py --config configs/pix2pix.yaml \
    --checkpoint checkpoints/pix2pix/generator_last.pt \
    --output results/tables/pix2pix_test_metrics.json

## Evaluate Diffusion (and Diffusion + Semantic Consistency)

`scripts/evaluate.py` currently supports single-forward-pass models (U-Net/Pix2Pix) directly. For diffusion, write a small custom loop using `GaussianDiffusion.p_sample_loop` as the `generate_fn` passed into `evaluate_model` — see `src/evaluation/evaluate.py` docstring.

In [ ]:
# TODO: custom diffusion evaluation loop using evaluate_model(..., generate_fn=...)

## FID (compute once, at the end — needs a reasonably large sample)

In [ ]:
# TODO: use pytorch-fid (already in requirements.txt) between a folder of
# generated EO images and a folder of real EO images from the test split.

## Semantic metrics (mIoU, F1, pixel accuracy)

Run the pretrained segmentation network (Stage 12) over generated vs. real EO test images and compare to land-cover ground truth using `src/evaluation/metrics.py` (`compute_confusion_stats`, `iou_from_confusion`, `pixel_accuracy_from_confusion`).

In [ ]:
# TODO: semantic evaluation loop.

## Aggregate into the final comparison table

Load each model's JSON results file from `results/tables/` and assemble the comparison table used in notebook 08 and the final report. Do not hand-type numbers — read them from the JSON files written above.

In [ ]:
import json, glob

rows = []
for path in sorted(glob.glob('results/tables/*_test_metrics.json')):
    with open(path) as f:
        data = json.load(f)
    rows.append({
        'model': path.split('/')[-1].replace('_test_metrics.json', ''),
        'psnr': data.get('psnr_mean'),
        'ssim': data.get('ssim_mean'),
        'lpips': data.get('lpips_mean'),
        'n': data.get('num_samples'),
    })

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv('results/tables/final_comparison.csv', index=False)
df